# Llama-3-Kor-BCCard-Finance-12B

- Hugging Face repo: `BCCard/Llama-3-Kor-BCCard-Finance-12B`
- Ollama model tag: `bc-llama3-finance-12b:q4`
- Local HF path: `C:\ollama-hf\llama3-bc-finance-12b`
- Default options used here: `temperature=0.6`, `top_p=0.9`, `num_ctx=4096`
- Note: generation_config.json has no temperature/top_p. Context is set conservatively for local q4 inference.


In [ ]:
import json
import urllib.error
import urllib.request
from pathlib import Path

OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "bc-llama3-finance-12b:q4"
HF_REPO = "BCCard/Llama-3-Kor-BCCard-Finance-12B"
LOCAL_MODEL_DIR = Path(r"C:\ollama-hf\llama3-bc-finance-12b")

DEFAULT_OPTIONS = {
    "temperature": 0.6,
    "top_p": 0.9,
    "num_ctx": 4096,
}

PROMPT = "비씨카드 연체가 발생했을 때 고객이 먼저 확인해야 할 사항을 설명해줘."

DOWNLOAD_CMD = 'powershell -ExecutionPolicy Bypass -File .\\scripts\\create_bccard_ollama_models.ps1 -BaseDir C:\\ollama-hf -Only Llama-3-Kor-BCCard-Finance-12B.ipynb -ValidateOnly -ContinueOnError'
CREATE_CMD = 'powershell -ExecutionPolicy Bypass -File .\\scripts\\create_bccard_gguf_q4_models.ps1 -Only bc-llama3-finance-12b:q4'


In [ ]:
def ollama_request(method: str, path: str, payload: dict | None = None, timeout: int = 180) -> dict:
    url = OLLAMA_BASE_URL.rstrip("/") + path
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8")
            return json.loads(raw) if raw else {}
    except urllib.error.URLError as exc:
        raise RuntimeError(f"Ollama server is not reachable at {OLLAMA_BASE_URL}. Start Ollama and rerun.") from exc


def installed_models() -> list[str]:
    data = ollama_request("GET", "/api/tags")
    return sorted(model.get("name", "") for model in data.get("models", []))


def ensure_model_ready() -> None:
    names = installed_models()
    print("Installed Ollama models:")
    for name in names:
        print(" -", name)

    if OLLAMA_MODEL in names:
        print(f"\nReady: {OLLAMA_MODEL}")
        return

    print(f"\nMissing Ollama model: {OLLAMA_MODEL}")
    print("\n1) Download the Hugging Face model:")
    print(DOWNLOAD_CMD)
    print("\n2) Register it in Ollama:")
    print(CREATE_CMD)
    raise SystemExit("Run the commands above, then rerun this notebook.")


In [ ]:
ensure_model_ready()

payload = {
    "model": OLLAMA_MODEL,
    "messages": [
        {"role": "user", "content": PROMPT},
    ],
    "stream": False,
    "options": DEFAULT_OPTIONS,
}

response = ollama_request("POST", "/api/chat", payload, timeout=600)
print(response.get("message", {}).get("content", ""))
